In [ ]:
print("[cell] setup", flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || true")
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.device_count(), flush=True)
PIP = os.environ.get("PIP_SPEC", '')   # --env PIP_SPEC=... / PIP_POST=... override the built-in install
if PIP and torch.cuda.is_available():
    sh(f"pip install -q {PIP}")
    if os.environ.get("PIP_POST", ''):
        sh(os.environ.get("PIP_POST", ''))
sh("pip list 2>/dev/null | grep -i -E '^(vllm|torch|transformers|qwen-asr|pandas|openpyxl) ' || true")


In [ ]:
print("[cell] write code files", flush=True)
FILES = {
    'common.py': '"""Shared helpers for GreenScan training on Kaggle (T4) and on a rented A100 (vast.ai).\n\nData is looked up in $GS_DATA, ./data, ../data, then /kaggle/input (a private dataset). Outputs go to $GS_OUT\n(default ./out). Every script prints "[cell] ..." at each stage and flushes, so a log shows where a run is.\n"""\nfrom __future__ import annotations\n\nimport json\nimport os\nimport random\nimport re\nimport time\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\nimport numpy as np\n\n\ndef cell(name: str) -> None:\n    print(f"[cell] {name}", flush=True)\n\n\ndef log(*a) -> None:\n    print(*a, flush=True)\n\n\ndef find(name: str) -> Path:\n    for root in (os.environ.get("GS_DATA", ""), "data", "../data", "/kaggle/input"):\n        if root and Path(root).exists():\n            hit = next(iter(sorted(Path(root).rglob(name))), None)\n            if hit:\n                return hit\n    raise FileNotFoundError(name)\n\n\ndef has(name: str) -> bool:\n    try:\n        find(name)\n        return True\n    except FileNotFoundError:\n        return False\n\n\ndef jl(path) -> list[dict]:\n    p = Path(path) if not isinstance(path, str) or Path(path).exists() else find(path)\n    return [json.loads(x) for x in open(p, encoding="utf-8") if x.strip()]\n\n\ndef wjl(path, rows) -> None:\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    with open(path, "w", encoding="utf-8") as fh:\n        for r in rows:\n            fh.write(json.dumps(r, ensure_ascii=False) + "\\n")\n\n\ndef out_dir(sub: str = "") -> Path:\n    p = Path(os.environ.get("GS_OUT", "out")) / sub\n    p.mkdir(parents=True, exist_ok=True)\n    return p\n\n\ndef save_json(path, obj) -> None:\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=1), encoding="utf-8")\n\n\ndef seed_all(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    try:\n        import torch\n        torch.manual_seed(seed)\n        torch.cuda.manual_seed_all(seed)\n    except ImportError:\n        pass\n\n\ndef norm60(s: str) -> str:\n    return re.sub(r"\\s+", " ", (s or "").lower()).strip()[:60]\n\n\ndef company_folds(groups: list[str], k: int = 5, seed: int = 13) -> list[tuple[list[int], list[int]]]:\n    """Out-of-company folds: every company sits in exactly one test fold; folds balanced by row count."""\n    count = Counter(groups)\n    comps = sorted(count, key=lambda c: (-count[c], c))\n    rng = random.Random(seed)\n    rng.shuffle(comps)\n    comps.sort(key=lambda c: -count[c])\n    load, fold_of = [0] * k, {}\n    for c in comps:\n        f = min(range(k), key=lambda i: load[i])\n        fold_of[c] = f\n        load[f] += count[c]\n    out = []\n    for f in range(k):\n        test = [i for i, g in enumerate(groups) if fold_of[g] == f]\n        train = [i for i, g in enumerate(groups) if fold_of[g] != f]\n        if test and train:\n            out.append((train, test))\n    return out\n\n\ndef prf(gold: list[str], pred: list[str], labels: list[str]) -> dict:\n    per = {}\n    for l in labels:\n        tp = sum(g == l and p == l for g, p in zip(gold, pred))\n        fp = sum(g != l and p == l for g, p in zip(gold, pred))\n        fn = sum(g == l and p != l for g, p in zip(gold, pred))\n        pr = tp / (tp + fp) if tp + fp else 0.0\n        rc = tp / (tp + fn) if tp + fn else 0.0\n        per[l] = {"precision": round(pr, 4), "recall": round(rc, 4),\n                  "f1": round(2 * pr * rc / (pr + rc), 4) if pr + rc else 0.0, "support": tp + fn, "predicted": tp + fp}\n    present = [l for l in labels if per[l]["support"]]\n    macro = sum(per[l]["f1"] for l in present) / len(present) if present else 0.0\n    acc = sum(g == p for g, p in zip(gold, pred)) / len(gold) if gold else 0.0\n    return {"macro_f1": round(macro, 4), "accuracy": round(acc, 4), "per_class": per, "n": len(gold)}\n\n\ndef confusion(gold: list[str], pred: list[str], labels: list[str]) -> list[list[int]]:\n    idx = {l: i for i, l in enumerate(labels)}\n    m = [[0] * len(labels) for _ in labels]\n    for g, p in zip(gold, pred):\n        if g in idx and p in idx:\n            m[idx[g]][idx[p]] += 1\n    return m\n\n\ndef boot_ci(items: list, groups: list[str], stat, n: int = 1000, seed: int = 7) -> list[float] | None:\n    """95% interval of stat(items) resampling whole companies (cluster bootstrap)."""\n    by = defaultdict(list)\n    for it, g in zip(items, groups):\n        by[g].append(it)\n    keys = sorted(by)\n    if len(keys) < 2:\n        return None\n    rng = random.Random(seed)\n    vals = []\n    for _ in range(n):\n        sample = [it for _ in keys for it in by[rng.choice(keys)]]\n        try:\n            vals.append(stat(sample))\n        except ZeroDivisionError:\n            continue\n    vals.sort()\n    return [round(vals[int(0.025 * len(vals))], 4), round(vals[int(0.975 * len(vals)) - 1], 4)] if vals else None\n\n\nclass Timer:\n    def __init__(self):\n        self.t0 = time.time()\n\n    def __call__(self) -> float:\n        return round(time.time() - self.t0, 1)\n',
    'train_t1.py': '"""T1 claim detector (offline product, CPU): multilingual-e5 sequence classifier, 4 classes, numbers masked.\n\n    python -u train_t1.py [--models e5-small,e5-base] [--losses ce,wce] [--folds 5]\n\nLabels: t1_train.jsonl (Claude, codebook v1.1 + Q13 v1.2 relabel). Sentences overlapping GS-Eval (first 60 normalised\ncharacters) are removed. Two losses:\n  ce   plain cross-entropy (the 11/11 recipe of s1_final.py)\n  wce  class-weighted cross-entropy (weights ∝ 1/sqrt(frequency)) + label smoothing 0.05\nReported: out-of-company 5-fold CV (macro-F1, claim recall), then a fit on all rows (2 seeds, probabilities averaged)\nscored on the registered S1 test: among GS-Eval sentences the product extracts, NOT_A_CLAIM caught and true claims\nwrongly flagged, for thresholds 0.5..0.95. Writes out/t1/metrics.json, out/t1/history.json, out/t1/<model>-<loss>-final/.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nimport os\nimport sys\nfrom collections import Counter\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom common import Timer, cell, company_folds, confusion, jl, log, norm60, out_dir, prf, save_json, seed_all  # noqa: E402\n\nLABELS = ["NOT_CLAIM", "QUALITATIVE_CLAIM", "QUANTIFIED_CLAIM", "VAGUE"]\nL2I = {l: i for i, l in enumerate(LABELS)}\nMODELS = {"e5-small": "intfloat/multilingual-e5-small", "e5-base": "intfloat/multilingual-e5-base"}\nHP = {"e5-small": dict(lr=5e-5, epochs=6), "e5-base": dict(lr=3e-5, epochs=6)}\nMAX_LEN, BATCH, SEEDS = 160, 16, [13, 29]\nTHRESHOLDS = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]\n\n\ndef fit(key, loss_kind, texts, labels, seed, hist=None, tag=""):\n    import torch\n    from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup\n    seed_all(seed)\n    dev = "cuda" if torch.cuda.is_available() else "cpu"\n    tok = AutoTokenizer.from_pretrained(MODELS[key])\n    model = AutoModelForSequenceClassification.from_pretrained(\n        MODELS[key], num_labels=4, id2label=dict(enumerate(LABELS)), label2id=L2I).float().to(dev)\n    hp = HP[key]\n    y = [L2I[l] for l in labels]\n    if loss_kind == "wce":\n        cnt = Counter(y)\n        w = torch.tensor([1 / math.sqrt(cnt.get(i, 1)) for i in range(4)], dtype=torch.float, device=dev)\n        lossf = torch.nn.CrossEntropyLoss(weight=w * 4 / w.sum(), label_smoothing=0.05)\n    else:\n        lossf = torch.nn.CrossEntropyLoss()\n    steps = hp["epochs"] * math.ceil(len(texts) / BATCH)\n    opt = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=0.01)\n    sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)\n    scaler = torch.amp.GradScaler(enabled=dev == "cuda")\n    import random\n    model.train()\n    for ep in range(hp["epochs"]):\n        order = list(range(len(texts)))\n        random.shuffle(order)\n        tot, n = 0.0, 0\n        for i in range(0, len(order), BATCH):\n            b = order[i:i + BATCH]\n            x = tok(["query: " + texts[j] for j in b], truncation=True, max_length=MAX_LEN, padding=True,\n                    return_tensors="pt").to(dev)\n            with torch.autocast(dev, dtype=torch.float16, enabled=dev == "cuda"):\n                loss = lossf(model(**x).logits.float(), torch.tensor([y[j] for j in b], device=dev))\n            opt.zero_grad()\n            scaler.scale(loss).backward()\n            scaler.step(opt)\n            scaler.update()\n            sch.step()\n            tot, n = tot + loss.item() * len(b), n + len(b)\n        if hist is not None:\n            hist.append({"run": tag, "epoch": ep + 1, "loss": round(tot / n, 4)})\n        log(f"[t1] {tag} epoch {ep + 1} loss {tot / n:.4f}")\n    return model.eval(), tok\n\n\ndef probs(model, tok, texts):\n    import torch\n    dev = next(model.parameters()).device\n    out = []\n    with torch.no_grad():\n        for i in range(0, len(texts), 64):\n            x = tok(["query: " + t for t in texts[i:i + 64]], truncation=True, max_length=MAX_LEN, padding=True,\n                    return_tensors="pt").to(dev)\n            out += torch.softmax(model(**x).logits.float(), -1).cpu().tolist()\n    return out\n\n\ndef gate(ev, p_not):\n    """S1 gate (llm_stages_v1): among extracted GS-Eval sentences, NOT_A_CLAIM caught / true claims flagged."""\n    rows = [(r["is_claim"], p) for r, p in zip(ev, p_not) if r.get("extracted") and r.get("is_claim") is not None]\n    out = {}\n    for t in THRESHOLDS:\n        out[str(t)] = {"not_claim_caught": sum((not c) and p >= t for c, p in rows),\n                       "not_claim_total": sum(not c for c, _ in rows),\n                       "true_claims_flagged": sum(c and p >= t for c, p in rows),\n                       "true_claims_total": sum(c for c, _ in rows)}\n    return out\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--models", default="e5-small,e5-base")\n    ap.add_argument("--losses", default="ce,wce")\n    ap.add_argument("--folds", type=int, default=5)\n    args = ap.parse_args()\n    import numpy as np\n    import torch\n\n    cell("t1 data")\n    train = jl("t1_train.jsonl")\n    ev = jl("s1_eval.jsonl")\n    if os.environ.get("LIMIT"):\n        train = train[: int(os.environ["LIMIT"])]\n    held = {r["norm60"] for r in ev}\n    kept = [r for r in train if norm60(r["text"]) not in held and r["label"] in L2I]\n    log(f"[t1] train {len(train)} -> {len(kept)} after GS-Eval overlap; labels {dict(Counter(r[\'label\'] for r in kept))}; "\n        f"q13 relabelled {sum(r[\'q13_relabelled\'] for r in kept)}, changed {sum(r[\'label\'] != r[\'label_v11\'] for r in kept)}")\n    texts = [r["text_masked"] for r in kept]\n    labels = [r["label"] for r in kept]\n    groups = [r["company_id"] for r in kept]\n    od = out_dir("t1")\n    metrics = {"train_rows": len(kept), "label_counts": dict(Counter(labels)), "runs": {}}\n    hist = []\n    for key in args.models.split(","):\n        for lk in args.losses.split(","):\n            name = f"{key}-{lk}"\n            cell(f"t1 cv {name}")\n            t = Timer()\n            oof = [None] * len(kept)\n            for f, (tr, te) in enumerate(company_folds(groups, args.folds)):\n                model, tok = fit(key, lk, [texts[i] for i in tr], [labels[i] for i in tr], SEEDS[0], hist, f"{name}-fold{f}")\n                for i, p in zip(te, probs(model, tok, [texts[i] for i in te])):\n                    oof[i] = p\n                del model\n                torch.cuda.empty_cache()\n            pred = [LABELS[int(np.argmax(p))] for p in oof]\n            cv = prf(labels, pred, LABELS)\n            is_c = [l != "NOT_CLAIM" for l in labels]\n            cv["claim_recall"] = round(sum(c and p != "NOT_CLAIM" for c, p in zip(is_c, pred)) / max(1, sum(is_c)), 4)\n            cv["confusion"] = confusion(labels, pred, LABELS)\n            cell(f"t1 final {name}")\n            acc = None\n            for s in SEEDS:\n                model, tok = fit(key, lk, texts, labels, s, hist, f"{name}-final-s{s}")\n                p = np.array(probs(model, tok, [r["text_masked"] for r in ev]))\n                acc = p if acc is None else acc + p\n                if s == SEEDS[0]:\n                    model.save_pretrained(od / f"{name}-final")\n                    tok.save_pretrained(od / f"{name}-final")\n                    import time\n                    cpu = model.to("cpu")\n                    torch.set_num_threads(4)\n                    t0 = time.time()\n                    probs(cpu, tok, [r["text_masked"] for r in ev[:64]])\n                    cpu_ms = round((time.time() - t0) / 64 * 1000, 1)\n                del model\n                torch.cuda.empty_cache()\n            acc /= len(SEEDS)\n            with open(od / f"s1_{name}.jsonl", "w", encoding="utf-8") as fh:   # per sentence, for error analysis\n                for r, pr in zip(ev, acc.tolist()):\n                    fh.write(json.dumps({"id": r["id"], "company": r.get("company"), "is_claim": r.get("is_claim"),\n                                         "extracted": r.get("extracted"), "text_masked": r["text_masked"],\n                                         "probs": dict(zip(LABELS, [round(x, 4) for x in pr]))}, ensure_ascii=False) + "\\n")\n            metrics["runs"][name] = {"cv": cv, "s1_gate": gate(ev, acc[:, 0].tolist()), "seconds": t(),\n                                     "cpu_ms_per_sentence_4threads": cpu_ms}\n            log(f"[t1] {name}: CV macro-F1 {cv[\'macro_f1\']} claim recall {cv[\'claim_recall\']} · gate@0.5 "\n                f"{metrics[\'runs\'][name][\'s1_gate\'][\'0.5\']} · {t()} s")\n            save_json(od / "metrics.json", metrics)\n            save_json(od / "history.json", hist)\n    log("[t1] DONE")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n',
}
for name, text in FILES.items():
    open(name, 'w', encoding='utf-8').write(text)
    print('wrote', name, len(text), flush=True)


In [ ]:
print('[cell] run 1: python -u train_t1.py --models e5-small,e5-base --losses wce', flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh('python -u train_t1.py --models e5-small,e5-base --losses wce')


In [ ]:
print("[cell] done", flush=True)
print("ALL CELLS OK", flush=True)